# Digital Communication Laboratory
## Experiment 10: Binary Phase Shift Keying (BPSK), Quadrature Phase Shift Keying (QPSK), and Differential Phase Shift Keying (DPSK)

**Programme:** B.Tech in Electronics and Communication Engineering  
**Platform:** Python 3 (NumPy, SciPy, Matplotlib, Pandas)  
**Mode:** Self-Contained First-Principles Simulation without Proprietary Toolboxes  

---

### Objectives & Syllabus Coverage:
1. **First-Principles Transceivers:** Implement modulators and coherent Euclidean slicers for BPSK and Gray-coded QPSK.
2. **QPSK Gray Mapping & Decision Boundaries:** Visualize the complex I-Q plane with quadrant decision regions and unit energy circle.
3. **Differential PSK (DBPSK):** Implement differential encoding and detection to overcome carrier phase ambiguity.
4. **Constellation Analysis:** Generate 2D scatter plots of ideal and noisy constellations across clean, $20\text{ dB}$, $10\text{ dB}$, and $5\text{ dB}$ SNR regimes.
5. **Phase Trajectories & Constellation Rotation:** Demonstrate progressive constellation rotation for $\Delta\theta \in \{0^\circ, 10^\circ, 30^\circ, 60^\circ, 90^\circ\}$.
6. **Decision Statistic Distributions:** Inspect empirical probability density functions (histograms) on $I$ and $Q$ rails under phase rotation.
7. **Monte Carlo BER Simulation:** Validate that BPSK and QPSK share identical BER curves $P_b = Q(\sqrt{2E_b/N_0})$ and quantify the $\approx 0.8\text{ dB}$ DBPSK penalty.
8. **Phase Sensitivity & Resilience:** Demonstrate DBPSK phase invariance vs catastrophic collapse of coherent BPSK/QPSK as phase offset increases from $0^\circ$ to $90^\circ$.
9. **Time-Domain Baseband Waveforms:** Plot In-Phase $I(t)$ and Quadrature $Q(t)$ pulse trains and symbol phase trajectories.
10. **Mandatory Syllabus Validation:** Manually map dibits `00`, `01`, `11`, `10` and verify receiver decisions before and after phase rotation.


In [ ]:
# Cell 1: Environment Setup, Library Imports, and Publication Styling
%matplotlib inline
import os
import sys
from pathlib import Path
from typing import Tuple, Dict, List, Optional

import numpy as np
import scipy.special as sp
import matplotlib.pyplot as plt
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

# Set plot styling for publication-quality visuals
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "lines.linewidth": 1.5,
    "grid.alpha": 0.5,
    "grid.linestyle": "--",
    "savefig.dpi": 300,
    "savefig.bbox": "tight"
})

# Core System Parameters (Strict Experiment Specifications)
RS = 1000.0                # Symbol rate = 1000 symbols/sec (1000 Baud)
TS = 1.0 / RS              # Symbol duration = 1.0 ms
ES = 1.0                   # Normalized average symbol energy = 1.0

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("[OK] Environment initialized. Parameters locked:")
print(f"    Symbol Rate Rs = {RS:.0f} Baud | Symbol Period Ts = {TS*1000:.1f} ms | Symbol Energy Es = {ES:.1f}")
print(f"    Outputs directory: {OUTPUT_DIR.resolve()}")


---
## 1. Modulation from First Principles: BPSK, Gray-Coded QPSK & DBPSK

- **BPSK:** $s_k = +1$ (bit 0), $-1$ (bit 1). $E_b = E_s = 1.0$.
- **Gray-Coded QPSK:** Transmits 2 bits per symbol. Normalized coordinates: $(\pm 1 \pm j)/\sqrt{2}$.
  - `00` $\to (+\frac{1}{\sqrt{2}}, +\frac{1}{\sqrt{2}})$ ($+45^\circ$)
  - `01` $\to (-\frac{1}{\sqrt{2}}, +\frac{1}{\sqrt{2}})$ ($+135^\circ$)
  - `11` $\to (-\frac{1}{\sqrt{2}}, -\frac{1}{\sqrt{2}})$ ($+225^\circ$)
  - `10` $\to (+\frac{1}{\sqrt{2}}, -\frac{1}{\sqrt{2}})$ ($+315^\circ$)
- **DBPSK:** Encodes information in phase transitions: $d_k = d_{k-1} \oplus b_k$.


In [ ]:
# Cell 2: First-Principles Modulators and Demodulators
def generate_random_bits(n_bits: int, seed: Optional[int] = 42) -> np.ndarray:
    rng = np.random.default_rng(seed)
    return rng.integers(0, 2, size=n_bits)

def modulate_bpsk(bits: np.ndarray) -> np.ndarray:
    return np.where(bits == 0, 1.0, -1.0)

def demodulate_bpsk(symbols: np.ndarray) -> np.ndarray:
    return np.where(np.real(symbols) > 0.0, 0, 1)

def modulate_qpsk_gray(bits: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    if len(bits) % 2 != 0:
        bits = bits[:-1]
    b0 = bits[0::2]
    b1 = bits[1::2]
    i_rail = np.where(b0 == 0, 1.0, -1.0)
    q_rail = np.where(b1 == 0, 1.0, -1.0)
    symbols = (i_rail + 1j * q_rail) / np.sqrt(2.0)
    return symbols, np.column_stack((b0, b1))

def demodulate_qpsk_gray(symbols: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    b0 = np.where(np.real(symbols) > 0.0, 0, 1)
    b1 = np.where(np.imag(symbols) > 0.0, 0, 1)
    demod_bits = np.empty(len(symbols) * 2, dtype=int)
    demod_bits[0::2] = b0
    demod_bits[1::2] = b1
    return demod_bits, np.column_stack((b0, b1))

def modulate_dbpsk(bits: np.ndarray) -> np.ndarray:
    n_bits = len(bits)
    d = np.zeros(n_bits + 1, dtype=int)
    for k in range(n_bits):
        d[k + 1] = d[k] ^ bits[k]
    return np.where(d == 0, 1.0, -1.0)

def demodulate_dbpsk(rx_symbols: np.ndarray) -> np.ndarray:
    u_k = rx_symbols[1:] * np.conj(rx_symbols[:-1])
    return np.where(np.real(u_k) > 0.0, 0, 1)

def apply_channel(symbols: np.ndarray, snr_db: float, phase_offset_deg: float = 0.0,
                  bits_per_symbol: int = 1, seed: Optional[int] = None) -> np.ndarray:
    eb = ES / bits_per_symbol
    snr_lin = 10.0 ** (snr_db / 10.0)
    n0 = eb / snr_lin
    sigma = np.sqrt(n0 / 2.0)
    rot = symbols * np.exp(1j * np.deg2rad(phase_offset_deg))
    rng = np.random.default_rng(seed)
    noise = rng.normal(0.0, sigma, size=len(symbols)) + 1j * rng.normal(0.0, sigma, size=len(symbols))
    return rot + noise

print("[OK] Modulator and channel models compiled successfully.")


---
## 2. QPSK Gray Mapping & Euclidean Slicing Decision Boundaries

In Gray coding, adjacent constellation quadrants differ by only a single bit (`00` $\to$ `01` $\to$ `11` $\to$ `10`). Consequently, any symbol error caused by crossing an adjacent decision boundary results in exactly **one bit error** ($P_b \approx P_s / 2$).


In [ ]:
# Cell 3: QPSK Gray Mapping and Decision Regions [Figure 2]
fig, ax = plt.subplots(figsize=(8.5, 8.5))

# Decision boundaries
ax.axhline(0, color="black", linewidth=2.0)
ax.axvline(0, color="black", linewidth=2.0)

# Quadrant shading (decision regions)
ax.fill_between([0, 2], 0, 2, color="blue", alpha=0.08, label="Region '00'")
ax.fill_between([-2, 0], 0, 2, color="orange", alpha=0.08, label="Region '01'")
ax.fill_between([-2, 0], -2, 0, color="green", alpha=0.08, label="Region '11'")
ax.fill_between([0, 2], -2, 0, color="purple", alpha=0.08, label="Region '10'")

# Unit circle representing Es = 1.0
circle = plt.Circle((0, 0), 1.0, color="gray", fill=False, linestyle=":", linewidth=1.5, label="Constant Energy $|s|=1$")
ax.add_artist(circle)

# Plot constellation points
norm = 1.0 / np.sqrt(2.0)
points = [
    (norm, norm, "00", 45, "blue"),
    (-norm, norm, "01", 135, "darkorange"),
    (-norm, -norm, "11", 225, "forestgreen"),
    (norm, -norm, "10", 315, "purple")
]

for x, y, label, angle, col in points:
    ax.plot(x, y, "o", markersize=14, color=col, markeredgecolor="black", markeredgewidth=1.5)
    ax.annotate(f"'{label}'\n({angle}°)", (x, y), textcoords="offset points",
                xytext=(15 if x > 0 else -35, 15 if y > 0 else -30),
                fontsize=12, fontweight="bold", color=col,
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=col, lw=1.5))
    ax.arrow(0, 0, x * 0.9, y * 0.9, head_width=0.05, head_length=0.05, fc=col, ec=col, alpha=0.7)

ax.set_title("QPSK Gray Mapping & Euclidean Slicing Decision Boundaries\n(Adjacent symbols differ by exactly 1 bit)", fontsize=13)
ax.set_xlabel("In-Phase Rail (Re)", fontsize=12)
ax.set_ylabel("Quadrature Rail (Im)", fontsize=12)
ax.set_xlim(-1.8, 1.8)
ax.set_ylim(-1.8, 1.8)
ax.grid(True, linestyle="--", alpha=0.5)
ax.legend(loc="upper right", framealpha=0.95)
ax.set_aspect("equal")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "2_qpsk_gray_mapping_decision_regions.png")
plt.show()


---
## 3. Mandatory Syllabus Validation: QPSK Gray Mapping & Demapper Decisions

> **Syllabus Requirement:** *"Manually map `00`, `01`, `11` and `10` and check the corresponding detector decisions."*

We evaluate the mapping and decision boundaries before and after introducing carrier phase rotations ($0^\circ, 30^\circ, 90^\circ$).


In [ ]:
# Cell 4: Mandatory Syllabus Validation - Mapping & Decision Checking [Figure 8]
def validate_qpsk_mapping() -> pd.DataFrame:
    dibits = np.array([[0, 0], [0, 1], [1, 1], [1, 0]])
    symbols, _ = modulate_qpsk_gray(dibits.flatten())
    records = []
    for i, dibit in enumerate(dibits):
        sym = symbols[i]
        d_str = f"{dibit[0]}{dibit[1]}"
        phase_deg = np.rad2deg(np.angle(sym))
        if phase_deg < 0:
            phase_deg += 360.0
        for theta in [0.0, 30.0, 90.0]:
            rx_sym = sym * np.exp(1j * np.deg2rad(theta))
            _, dem_dibits = demodulate_qpsk_gray(np.array([rx_sym]))
            dec_str = f"{dem_dibits[0, 0]}{dem_dibits[0, 1]}"
            records.append({
                "Tx Dibit": d_str,
                "Ideal Symbol": f"{np.real(sym):+.4f} {np.imag(sym):+.4f}j",
                "Ideal Phase": f"{phase_deg:.1f}°",
                "Phase Offset": f"{theta:.0f}°",
                "Received Symbol": f"{np.real(rx_sym):+.4f} {np.imag(rx_sym):+.4f}j",
                "Rx Decision": dec_str,
                "Status": "CORRECT" if dec_str == d_str else "DECISION ERROR"
            })
    return pd.DataFrame(records)

df_map = validate_qpsk_mapping()
display(df_map)

# Plot Figure 8: Mandatory validation mapping diagram and verification table
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
norm = 1.0 / np.sqrt(2.0)
ideals = {
    "00": (norm, norm, "blue"),
    "01": (-norm, norm, "darkorange"),
    "11": (-norm, -norm, "forestgreen"),
    "10": (norm, -norm, "purple")
}

for dibit, (x, y, col) in ideals.items():
    ax1.plot(x, y, "o", markersize=12, color=col, markeredgecolor="black", label=f"Tx '{dibit}' (Ideal)")
    rot_30 = (x + 1j * y) * np.exp(1j * np.deg2rad(30.0))
    ax1.plot(np.real(rot_30), np.imag(rot_30), "^", markersize=9, color=col, alpha=0.6)
    rot_90 = (x + 1j * y) * np.exp(1j * np.deg2rad(90.0))
    ax1.plot(np.real(rot_90), np.imag(rot_90), "x", markersize=10, markeredgewidth=2, color="red")

ax1.axhline(0, color="black", linestyle="--", linewidth=1.5)
ax1.axvline(0, color="black", linestyle="--", linewidth=1.5)
ax1.set_title("Mandatory Validation: Dibits '00', '01', '11', '10'\n[O = Ideal, ^ = +30°, X = +90° Rotated]", fontsize=12)
ax1.set_xlabel("In-Phase Rail (Re)")
ax1.set_ylabel("Quadrature Rail (Im)")
ax1.set_xlim(-1.6, 1.6); ax1.set_ylim(-1.6, 1.6); ax1.grid(True); ax1.set_aspect("equal")

ax2.axis("off")
table_data = []
for _, r in df_map.iterrows():
    table_data.append([
        r["Tx Dibit"],
        r["Ideal Phase"],
        r["Phase Offset"],
        r["Received Symbol"],
        r["Rx Decision"],
        r["Status"]
    ])
col_labels = ["Tx Dibit", "Ideal Phase", "Offset", "Rx Symbol (I + jQ)", "Rx Decision", "Status"]
table = ax2.table(cellText=table_data, colLabels=col_labels, loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1.15, 1.55)
ax2.set_title("Detector Decision Verification Across Phase Offsets", fontsize=12, pad=15)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "8_mandatory_mapping_validation.png")
plt.show()


---
## 4. Time-Domain Baseband I-Q Waveforms & Symbol Phase Trajectories

Visualizing the continuous-time pulse transitions on the in-phase $I(t)$ and quadrature $Q(t)$ rails alongside the instantaneous symbol phase trajectory $\theta(t) \in \{45^\circ, 135^\circ, 225^\circ, 315^\circ\}$.


In [ ]:
# Cell 5: Time-Domain In-Phase & Quadrature Waveforms [Figure 7]
demo_wave_bits = np.array([0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0])
qpsk_wave_syms, dibits_wave = modulate_qpsk_gray(demo_wave_bits)
n_syms_wave = len(qpsk_wave_syms)

sps = 50
t = np.arange(n_syms_wave * sps) / (sps * RS)
i_wave = np.repeat(np.real(qpsk_wave_syms), sps)
q_wave = np.repeat(np.imag(qpsk_wave_syms), sps)
phase_wave = np.repeat(np.rad2deg(np.angle(qpsk_wave_syms)), sps)

fig, axes = plt.subplots(3, 1, figsize=(13, 8.5), sharex=True)

# I-Rail
axes[0].plot(t * 1000, i_wave, color="royalblue", linewidth=2.0)
axes[0].set_ylabel("In-Phase $I(t)$")
axes[0].set_title("QPSK Baseband In-Phase Component ($b_0$: 0 -> +0.707, 1 -> -0.707)")
axes[0].set_ylim(-1.0, 1.0); axes[0].grid(True)

# Q-Rail
axes[1].plot(t * 1000, q_wave, color="darkorange", linewidth=2.0)
axes[1].set_ylabel("Quadrature $Q(t)$")
axes[1].set_title("QPSK Baseband Quadrature Component ($b_1$: 0 -> +0.707, 1 -> -0.707)")
axes[1].set_ylim(-1.0, 1.0); axes[1].grid(True)

# Symbol Phase
axes[2].step(t * 1000, phase_wave, where="post", color="forestgreen", linewidth=2.0)
axes[2].set_ylabel("Phase (Degrees)")
axes[2].set_xlabel("Time (milliseconds)")
axes[2].set_title("QPSK Symbol Phase Trajectory: [45°, 135°, 225°, 315°]")
axes[2].set_ylim(-190, 190); axes[2].grid(True)

for i in range(n_syms_wave):
    d_str = f"{dibits_wave[i, 0]}{dibits_wave[i, 1]}"
    for ax in axes:
        ax.axvline(i * TS * 1000, color="black", linestyle=":", alpha=0.5)
    axes[0].text((i + 0.5) * TS * 1000, 0.0, f"'{d_str}'", ha="center", va="center",
                 fontsize=11, fontweight="bold", bbox=dict(boxstyle="square", fc="white", alpha=0.7))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "7_time_domain_iq_waveforms.png")
plt.show()


---
## 5. Constellation Analysis: Ideal vs Noisy AWGN Clouds

Visualizing BPSK and QPSK constellations as channel SNR decreases from clean ($+\infty\text{ dB}$) down to $20\text{ dB}$, $10\text{ dB}$, and $5\text{ dB}$.


In [ ]:
# Cell 6: Constellation Scatter Plots across SNR Regimes [Figure 1]
n_syms = 2000
bits = generate_random_bits(n_syms * 2, seed=42)
bpsk_s = modulate_bpsk(bits[:n_syms])
qpsk_s, _ = modulate_qpsk_gray(bits)

snr_levels = [20.0, 10.0, 5.0]
fig, axes = plt.subplots(2, 4, figsize=(15, 7.5))

# (Row 1): BPSK
axes[0, 0].scatter(np.real(bpsk_s[:200]), np.imag(bpsk_s[:200]), color="red", s=40, label="Symbols")
axes[0, 0].set_title("BPSK: Ideal Constellation")

for i, snr in enumerate(snr_levels):
    rx = apply_channel(bpsk_s, snr, bits_per_symbol=1, seed=100 + i)
    axes[0, i + 1].scatter(np.real(rx), np.imag(rx), color="crimson", alpha=0.3, s=8)
    axes[0, i + 1].set_title(f"BPSK: AWGN ({snr:.0f} dB SNR)")

# (Row 2): QPSK
axes[1, 0].scatter(np.real(qpsk_s[:200]), np.imag(qpsk_s[:200]), color="blue", s=40, label="Symbols")
axes[1, 0].set_title("QPSK: Ideal Gray Constellation")

for i, snr in enumerate(snr_levels):
    rx = apply_channel(qpsk_s, snr, bits_per_symbol=2, seed=200 + i)
    axes[1, i + 1].scatter(np.real(rx), np.imag(rx), color="navy", alpha=0.3, s=8)
    axes[1, i + 1].set_title(f"QPSK: AWGN ({snr:.0f} dB SNR)")

for row in axes:
    for ax in row:
        ax.axhline(0, color="black", linestyle="--", alpha=0.6)
        ax.axvline(0, color="black", linestyle="--", alpha=0.6)
        ax.set_xlim(-2.0, 2.0); ax.set_ylim(-2.0, 2.0); ax.grid(True); ax.set_aspect("equal")
        ax.set_xlabel("In-Phase (I)"); ax.set_ylabel("Quadrature (Q)")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "1_constellations_ideal_and_noisy.png")
plt.show()


---
## 6. Carrier Phase Offset & Constellation Rotation

Phase rotations $\Delta\theta \in \{0^\circ, 10^\circ, 30^\circ, 60^\circ, 90^\circ\}$ rotate the entire constellation cloud.
- At $\Delta\theta = 45^\circ$, constellation points align directly with the decision axes, destroying the noise margin.
- At $\Delta\theta = 90^\circ$, points rotate into neighboring quadrants, causing systematic quadrant inversion.


In [ ]:
# Cell 7: Constellation Rotation Sequence [Figure 3]
offsets = [0.0, 10.0, 30.0, 60.0, 90.0]
fig, axes = plt.subplots(1, 5, figsize=(18, 4.2), sharey=True)

for i, po in enumerate(offsets):
    rx = apply_channel(qpsk_s, snr_db=18.0, phase_offset_deg=po, bits_per_symbol=2, seed=800 + i)
    axes[i].scatter(np.real(rx), np.imag(rx), color="royalblue", alpha=0.35, s=12)
    axes[i].axhline(0, color="black", linestyle="--", linewidth=1.2)
    axes[i].axvline(0, color="black", linestyle="--", linewidth=1.2)
    rot_ideals = (np.array([1+1j, -1+1j, -1-1j, 1-1j]) / np.sqrt(2.0)) * np.exp(1j * np.deg2rad(po))
    axes[i].scatter(np.real(rot_ideals), np.imag(rot_ideals), color="red", marker="x", s=80, linewidth=2.5)
    axes[i].set_title(f"Phase Offset $\\Delta\\theta = {po:.0f}^\\circ$", fontsize=12)
    axes[i].set_xlabel("In-Phase (I)")
    if i == 0:
        axes[i].set_ylabel("Quadrature (Q)")
    axes[i].set_xlim(-1.8, 1.8); axes[i].set_ylim(-1.8, 1.8); axes[i].grid(True); axes[i].set_aspect("equal")

fig.suptitle("Impact of Carrier Phase Offset on QPSK Constellation & Decision Boundaries (AWGN SNR = 18 dB)", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "3_phase_trajectories_and_constellation_rotation.png")
plt.show()


---
## 7. Decision Statistic Distributions under Phase Error

To understand decision degradation, we inspect the empirical histograms of in-phase ($I$) and quadrature ($Q$) projections under $\Delta\theta = 0^\circ$ (clean margin) vs $\Delta\theta = 30^\circ$ (severe cloud spread).


In [ ]:
# Cell 8: Decision Statistic Distributions under Phase Error [Figure 4]
n_hist_syms = 8000
hist_bits = generate_random_bits(n_hist_syms * 2, seed=999)
qpsk_hist_syms, _ = modulate_qpsk_gray(hist_bits)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

rx_0 = apply_channel(qpsk_hist_syms, snr_db=10.0, phase_offset_deg=0.0, bits_per_symbol=2, seed=101)
rx_30 = apply_channel(qpsk_hist_syms, snr_db=10.0, phase_offset_deg=30.0, bits_per_symbol=2, seed=102)

# I-Rail at 0 deg
axes[0, 0].hist(np.real(rx_0[hist_bits[0::2] == 0]), bins=50, density=True, alpha=0.6, color="blue", label="Bit 0 ($I > 0$)")
axes[0, 0].hist(np.real(rx_0[hist_bits[0::2] == 1]), bins=50, density=True, alpha=0.6, color="red", label="Bit 1 ($I < 0$)")
axes[0, 0].axvline(0, color="black", linestyle="--", linewidth=1.8, label="Decision Boundary")
axes[0, 0].set_title(r"In-Phase Rail ($\Delta\theta = 0^\circ$ Offset - Pristine Margin)")
axes[0, 0].set_ylabel("Probability Density"); axes[0, 0].grid(True); axes[0, 0].legend(loc="upper right")

# Q-Rail at 0 deg
axes[0, 1].hist(np.imag(rx_0[hist_bits[1::2] == 0]), bins=50, density=True, alpha=0.6, color="blue", label="Bit 0 ($Q > 0$)")
axes[0, 1].hist(np.imag(rx_0[hist_bits[1::2] == 1]), bins=50, density=True, alpha=0.6, color="red", label="Bit 1 ($Q < 0$)")
axes[0, 1].axvline(0, color="black", linestyle="--", linewidth=1.8, label="Decision Boundary")
axes[0, 1].set_title(r"Quadrature Rail ($\Delta\theta = 0^\circ$ Offset - Pristine Margin)")
axes[0, 1].grid(True); axes[0, 1].legend(loc="upper right")

# I-Rail at 30 deg
axes[1, 0].hist(np.real(rx_30[hist_bits[0::2] == 0]), bins=50, density=True, alpha=0.6, color="blue", label="Bit 0")
axes[1, 0].hist(np.real(rx_30[hist_bits[0::2] == 1]), bins=50, density=True, alpha=0.6, color="red", label="Bit 1")
axes[1, 0].axvline(0, color="black", linestyle="--", linewidth=1.8, label="Decision Boundary")
axes[1, 0].set_title(r"In-Phase Rail ($\Delta\theta = 30^\circ$ Offset - Severe Cloud Spread)")
axes[1, 0].set_xlabel("Re(y) Decision Statistic"); axes[1, 0].set_ylabel("Probability Density"); axes[1, 0].grid(True); axes[1, 0].legend(loc="upper right")

# Q-Rail at 30 deg
axes[1, 1].hist(np.imag(rx_30[hist_bits[1::2] == 0]), bins=50, density=True, alpha=0.6, color="blue", label="Bit 0")
axes[1, 1].hist(np.imag(rx_30[hist_bits[1::2] == 1]), bins=50, density=True, alpha=0.6, color="red", label="Bit 1")
axes[1, 1].axvline(0, color="black", linestyle="--", linewidth=1.8, label="Decision Boundary")
axes[1, 1].set_title(r"Quadrature Rail ($\Delta\theta = 30^\circ$ Offset - Cross-Talk Spillover)")
axes[1, 1].set_xlabel("Im(y) Decision Statistic"); axes[1, 1].grid(True); axes[1, 1].legend(loc="upper right")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "4_decision_statistic_histograms.png")
plt.show()


---
## 8. Monte Carlo Bit Error Rate (BER) Simulation vs Theory

We compare:
1. **Coherent BPSK & QPSK:** $P_b = Q(\sqrt{2 E_b/N_0})$ (Identical curves!)
2. **DBPSK:** $P_b = \frac{1}{2} \exp(-E_b/N_0)$
3. **Coherent BPSK with Phase Offset $\theta$:** $P_b = Q(\sqrt{2 E_b/N_0}\cos\theta)$
4. **DBPSK with Phase Offset:** Completely invariant to static phase offsets!


In [ ]:
# Cell 9: Fast Vectorized BER Simulation vs Theory [Figure 5]
eb_range = np.linspace(0.0, 12.0, 13)
n_bits_sim = 80000
sim_bits = generate_random_bits(n_bits_sim, seed=123)
if len(sim_bits) % 2 != 0:
    sim_bits = sim_bits[:-1]

bpsk_tx = modulate_bpsk(sim_bits)
qpsk_tx, _ = modulate_qpsk_gray(sim_bits)
dbpsk_tx = modulate_dbpsk(sim_bits)

ber_b_ideal = np.zeros(len(eb_range))
ber_q_ideal = np.zeros(len(eb_range))
ber_d_ideal = np.zeros(len(eb_range))
ber_b_10 = np.zeros(len(eb_range))
ber_b_30 = np.zeros(len(eb_range))
ber_d_30 = np.zeros(len(eb_range))

for idx, eb_db in enumerate(eb_range):
    # Ideal BPSK & QPSK
    rx_b = apply_channel(bpsk_tx, eb_db, 0.0, 1, 1000 + idx)
    ber_b_ideal[idx] = np.mean(demodulate_bpsk(rx_b) != sim_bits)
    rx_q = apply_channel(qpsk_tx, eb_db, 0.0, 2, 2000 + idx)
    q_b, _ = demodulate_qpsk_gray(rx_q)
    ber_q_ideal[idx] = np.mean(q_b != sim_bits)

    # DBPSK Ideal
    rx_d = apply_channel(dbpsk_tx, eb_db, 0.0, 1, 3000 + idx)
    ber_d_ideal[idx] = np.mean(demodulate_dbpsk(rx_d) != sim_bits)

    # Phase Offset 10 deg & 30 deg
    rx_b10 = apply_channel(bpsk_tx, eb_db, 10.0, 1, 3500 + idx)
    ber_b_10[idx] = np.mean(demodulate_bpsk(rx_b10) != sim_bits)
    rx_b30 = apply_channel(bpsk_tx, eb_db, 30.0, 1, 4000 + idx)
    ber_b_30[idx] = np.mean(demodulate_bpsk(rx_b30) != sim_bits)
    rx_d30 = apply_channel(dbpsk_tx, eb_db, 30.0, 1, 5000 + idx)
    ber_d_30[idx] = np.mean(demodulate_dbpsk(rx_d30) != sim_bits)

eb_fine = np.linspace(0.0, 12.0, 100)
eb_lin = 10.0 ** (eb_fine / 10.0)
th_bpsk = 0.5 * sp.erfc(np.sqrt(eb_lin))
th_dbpsk = 0.5 * np.exp(-eb_lin)
th_bpsk_10 = 0.5 * sp.erfc(np.sqrt(eb_lin) * np.cos(np.deg2rad(10.0)))
th_bpsk_30 = 0.5 * sp.erfc(np.sqrt(eb_lin) * np.cos(np.deg2rad(30.0)))

fig, ax = plt.subplots(figsize=(11, 7.5))

ax.semilogy(eb_fine, th_bpsk, "b-", linewidth=2.0, label="Coherent BPSK / QPSK (Theory)")
ax.semilogy(eb_range, ber_b_ideal, "bo", markersize=6, fillstyle="none", markeredgewidth=1.8, label="Coherent BPSK (Sim)")
ax.semilogy(eb_range, ber_q_ideal, "b^", markersize=6, fillstyle="none", markeredgewidth=1.8, label="Coherent QPSK (Sim)")

ax.semilogy(eb_fine, th_dbpsk, "g-", linewidth=1.8, label="DBPSK (Theory)")
ax.semilogy(eb_range, ber_d_ideal, "gs", markersize=6, fillstyle="none", markeredgewidth=1.8, label="DBPSK (Sim)")

ax.semilogy(eb_fine, th_bpsk_10, "m--", linewidth=1.5, label=r"BPSK $\Delta\theta=10^\circ$ (Theory)")
ax.semilogy(eb_range, ber_b_10, "md", markersize=5, fillstyle="none", label=r"BPSK $\Delta\theta=10^\circ$ (Sim)")

ax.semilogy(eb_fine, th_bpsk_30, "r--", linewidth=1.5, label=r"BPSK $\Delta\theta=30^\circ$ (Theory)")
ax.semilogy(eb_range, ber_b_30, "rx", markersize=6, markeredgewidth=1.8, label=r"BPSK $\Delta\theta=30^\circ$ (Sim)")

ax.semilogy(eb_range, ber_d_30, "k+", markersize=7, markeredgewidth=1.8, label=r"DBPSK $\Delta\theta=30^\circ$ (Sim - Phase Immune!)")

ax.set_title("Bit Error Rate (BER) for BPSK, QPSK, and DBPSK with Carrier Phase Offsets", fontsize=13)
ax.set_xlabel(r"Energy per Bit to Noise Spectral Density $E_b / N_0$ (dB)", fontsize=12)
ax.set_ylabel("Bit Error Rate (BER)", fontsize=12)
ax.set_xlim(0, 12); ax.set_ylim(1e-5, 0.6); ax.grid(True, which="both", linestyle="--", alpha=0.6)
ax.legend(loc="lower left", framealpha=0.95)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "5_ber_vs_ebno_comparison.png")
plt.show()


---
## 9. BER Degradation vs Phase Offset Sweep

Evaluating the sensitivity of BPSK, QPSK, and DBPSK to continuous carrier phase offsets $\Delta\theta \in [0^\circ, 90^\circ]$ at fixed $E_b/N_0 = 8\text{ dB}$.


In [ ]:
# Cell 10: BER Degradation vs Continuous Phase Offset [Figure 6]
phase_angles = np.linspace(0.0, 90.0, 19)
snr_db = 8.0
n_bits_po = 80000
bits_po = generate_random_bits(n_bits_po, seed=54321)
if len(bits_po) % 2 != 0:
    bits_po = bits_po[:-1]

bpsk_po_syms = modulate_bpsk(bits_po)
qpsk_po_syms, _ = modulate_qpsk_gray(bits_po)
dbpsk_po_syms = modulate_dbpsk(bits_po)

ber_bpsk_sweep = np.zeros(len(phase_angles))
ber_qpsk_sweep = np.zeros(len(phase_angles))
ber_dbpsk_sweep = np.zeros(len(phase_angles))

for i, po in enumerate(phase_angles):
    rx_b = apply_channel(bpsk_po_syms, snr_db, phase_offset_deg=po, bits_per_symbol=1, seed=300 + i)
    ber_bpsk_sweep[i] = np.mean(demodulate_bpsk(rx_b) != bits_po)

    rx_q = apply_channel(qpsk_po_syms, snr_db, phase_offset_deg=po, bits_per_symbol=2, seed=400 + i)
    q_bits, _ = demodulate_qpsk_gray(rx_q)
    ber_qpsk_sweep[i] = np.mean(q_bits != bits_po)

    rx_d = apply_channel(dbpsk_po_syms, snr_db, phase_offset_deg=po, bits_per_symbol=1, seed=500 + i)
    ber_dbpsk_sweep[i] = np.mean(demodulate_dbpsk(rx_d) != bits_po)

fig, ax = plt.subplots(figsize=(11, 6.5))

ax.semilogy(phase_angles, ber_bpsk_sweep, "ro-", linewidth=2.0, label=f"Coherent BPSK ($E_b/N_0={snr_db}$ dB)")
ax.semilogy(phase_angles, ber_qpsk_sweep, "bs-", linewidth=2.0, label=f"Coherent QPSK ($E_b/N_0={snr_db}$ dB)")
ax.semilogy(phase_angles, ber_dbpsk_sweep, "g^--", linewidth=2.2, label=f"DBPSK ($E_b/N_0={snr_db}$ dB - Invariant!)")

ax.axvline(45.0, color="purple", linestyle=":", linewidth=1.5, label=r"QPSK Decision Boundary Collapse ($45^\circ$)")
ax.axvline(90.0, color="crimson", linestyle=":", linewidth=1.5, label=r"BPSK Complete Collapse ($90^\circ$)")

ax.set_title(f"Bit Error Rate Degradation vs Carrier Phase Offset ($E_b/N_0 = {snr_db}$ dB)", fontsize=13)
ax.set_xlabel(r"Carrier Phase Offset $\Delta\theta$ (Degrees)", fontsize=12)
ax.set_ylabel("Bit Error Rate (BER)", fontsize=12)
ax.set_xlim(0, 90); ax.set_ylim(1e-4, 0.6); ax.grid(True, which="both", linestyle="--", alpha=0.6)
ax.legend(loc="center left", framealpha=0.95)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "6_ber_degradation_vs_phase_offset.png")
plt.show()


---
## 10. Conclusions & Laboratory Findings

1. **BPSK vs QPSK Spectral & Power Efficiency:**
   - QPSK transmits $2\text{ bits/symbol}$, achieving double the data rate in the exact same RF bandwidth as BPSK.
   - Coherent QPSK achieves the identical BER as BPSK in AWGN ($P_b = Q(\sqrt{2E_b/N_0})$) because in-phase and quadrature branches are completely orthogonal.
2. **Gray Coding Advantage:**
   - Adjacent constellation quadrants differ by only a single bit, ensuring single bit errors for symbol decision crossings.
3. **Phase Ambiguity Resilience via DBPSK:**
   - DBPSK encodes data in phase differences ($U_k = y_k y_{k-1}^*$), rendering demodulation completely invariant to static carrier phase errors at the cost of a modest $\approx 0.8\text{ dB}$ power penalty.
4. **All 8 Publication Figures:**
   - All figures generated and saved to `outputs/` in 300 DPI publication quality.
